# CYGNSS anomaly visualization with precipitation data

Import the necessary libraries for working with Google Earth Engine (GEE) and geemap.

In [1]:
import ee
import geemap
import cmocean
import numpy as np
import matplotlib as mpl

SET PROJECT NAME, authenticate and set up your Earth Engine environment by creating a Google Cloud Project and enabling the Earth Engine API (https://www.google.com/url?q=https%3A%2F%2Fconsole.cloud.google.com%2Fapis%2Fapi%2Fearthengine.googleapis.com), if you haven’t already done so. Detailed setup instructions are available at https://developers.google.com/earth-engine/guides/access#a-role-in-a-cloud-project

In [2]:
project_name = 'YOUR_EE_PROJECTNAME'
ee.Authenticate()
ee.Initialize(project=project_name) 

Load the CYGNSS collection

In [3]:
cygnss_collection = ee.ImageCollection("projects/sat-io/open-datasets/CYGNSS/inundation")


Set up the functions to (i) remove NaN values from CYGNSS data, (ii) convert inundation data to fractional inundation, and (iii) remove bias from the fractional inundation data.

In [4]:
def mask_anomalies(image):
    image = image.updateMask(image.neq(255))  # mask no-data values (255)
    image = image.subtract(100)               # remove bias
    return image.divide(100)                  # scale to fractional values

Select an image and band from the CYGNSS collection.
Mask no-data values, convert the data to fractional values, and remove bias.

In [5]:
date = '2025-11-07'
cygnss_image = cygnss_collection.filterDate(date, ee.Date(date).advance(1, 'day')).first() 

inundation_anomalies = mask_anomalies(cygnss_image.select('inundation_anomaly_3day')) # selecting the appropriate band and applying a function to mask no-data values, removing bias, and scale to fractional inundation

Select precipitation data from the Earth Engine Data Catalog. Calculate accumulated precipitation for the date range of interest


In [6]:
gpm = ee.ImageCollection("NASA/GPM_L3/IMERG_V07")

accumulated_rain = gpm \
    .filterDate('2025-11-04', '2025-11-08') \
    .select('precipitation') \
    .map(lambda img: img.multiply(0.5)) \
    .sum()

Set the visualization parameters (colors and value ranges) for the plotted bands. You can define your own colormap. By default, GEE accepts HEX or CSS color codes. The code below converts Matplotlib colormaps to HEX color codes. Separate visualization parameters are defined for inundation anomaly data (nominal range: −1 to 1) and precipitation data.

In [7]:
# Inundation anomalies palette and visualisation params
anomaly_palette = cmocean.cm.balance_r(np.linspace(0, 1, 256)) #palette (balance) from matplotlib (https://matplotlib.org/cmocean/)
anomaly_palette_hex = [mpl.colors.rgb2hex(color) for color in anomaly_palette]

anomaly_vis = {
    'min': -0.2,  
    'max': 0.2,  
    'palette': anomaly_palette_hex  # Paleta kolorów dla anomalii (niebieski-czerwony)
}

#Rain palette and visualisation params
rain_palette = cmocean.cm.rain(np.linspace(0, 1, 256)) #palette (rain) from matplotlib (https://matplotlib.org/cmocean/)
rain_palette_hex = [mpl.colors.rgb2hex(color) for color in rain_palette]

rain_vis = {
    'min': 0,  
    'max': 80,  
    'palette': rain_palette_hex  
}

Plot the region of interest with inundation anomalies and precipitation data. Please note that when plotting observational data, which may contain a large number of NoData pixels, it may be necessary to zoom in to properly visualize the colors.

In [8]:
# Region of interest – South America
roi = ee.Geometry.Rectangle([
    -77, -40,   # west, south
    -40, -20     # east, north
])

Map = geemap.Map() # generate map
Map.add_basemap('HYBRID') # add the 'HYBRID' layer containing satellite imagery
Map.addLayer(inundation_anomalies, anomaly_vis, "Inundation Anomaly") # Add layer with inundation anomalies using the anomaly_vis params
Map.addLayer(accumulated_rain, rain_vis, "3-day Rainfall", opacity=0.6) # Add layer with Rainfall data
Map.add_colorbar(anomaly_vis, label="Inundation Anomaly", orientation="vertical", position="bottomleft") # Add the colorbar for inundation data on the left.
Map.add_colorbar(rain_vis, label="Accumulated rain [mm]", orientation="vertical", position="bottomright") # Add the colorbar for precipitation data on the right.
Map.center_object(roi, 5) # Set the initial view to the ROI.
Map # Display the map. Without this command, the map will not be rendered.


Map(center=[-30.91366966039448, -58.5], controls=(WidgetControl(options=['position', 'transparent_bg'], widget…